In [ ]:
'''
Cell 1   → Mount Drive
Cell 1B  → Restore code
Cell 5   → Install packages
Cell 6   → GPU check
Cell 8   → Sanity check
Cell 9   → Config
Cell 11  → Resume training

or

full resume cell
cell 9
cell 11
'''

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ============================================================
# FULL RESUME CELL — run this every new session, then Cell 11
# ============================================================
import shutil, os, sys, torch

# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Restore code
BASE = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'
src  = f'{BASE}/code'
dst  = '/content/conv_mae_mri'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
sys.path.insert(0, dst)

# 3. Fix bugs in dataset.py
path = f'{dst}/data/dataset.py'
code = open(path).read()
code = code.replace('if not img_path.exists():', 'if img_path is None:')
open(path, 'w').write(code)

# 4. Fix PyTorch safe globals
from utils.config import Config
torch.serialization.add_safe_globals([Config])

# 5. Install packages
os.system("pip install -q torch torchvision SimpleITK nibabel scikit-image scikit-learn pytorch-msssim antspyx monai matplotlib tqdm")

# 6. Verify checkpoint
# ckpt = f'{BASE}/checkpoints/conv_mae_last.pth'
# state = torch.load(ckpt, map_location='cpu', weights_only=False)
# print(f"\n✓ Everything ready")
# print(f"✓ Will resume from epoch {state['epoch']+1}")
# print(f"✓ Best val_loss so far: {state['val_loss']:.5f}")
# print(f"\nNow run Cell 9 (config) then Cell 11 (trainer)")

Mounted at /content/drive


0

In [ ]:
#CELL 1

from google.colab import drive
drive.mount('/content/drive',force_remount=True)
print("✓ Drive mounted")

Mounted at /content/drive
✓ Drive mounted


In [ ]:
#CELL 1b
import shutil, os, sys

BASE = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'
src  = f'{BASE}/code'
dst  = '/content/conv_mae_mri'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
sys.path.insert(0, dst)
print("✓ Project restored:", os.listdir(dst))

✓ Project restored: ['models', 'training', 'notebooks', 'README.md', '{data,models,training,evaluation,utils,notebooks}', '__init__.py', 'utils', 'data', 'evaluation']


In [ ]:
#CELL 1c
path = '/content/conv_mae_mri/data/dataset.py'
code = open(path).read()

# Bug 1: img_path None check
code = code.replace(
    'if not img_path.exists():',
    'if img_path is None:'
)

# Bug 2: test_loader num_workers hardcoded to 2
drive_path = f'{BASE}/code/data/dataset.py'
open(path, 'w').write(code)
open(drive_path, 'w').write(code)

print("✓ Bugs fixed in dataset.py")

# Verify the cap is 150
if 'all_files[:150]' in code:
    print("✓ IXI cap is 150")
elif 'all_files[:60]' in code:
    print("✗ Cap still 60 — fix it on Drive manually")
else:
    print("✓ No cap found (using all files)")

✓ Bugs fixed in dataset.py
✓ IXI cap is 150


In [ ]:
# Cell 1D — Fix checkpoint loading for newer PyTorch
import torch
import sys
sys.path.insert(0, '/content/conv_mae_mri')

from utils.config import Config

# Tell PyTorch it's safe to load our Config class
torch.serialization.add_safe_globals([Config])
print("✓ Safe globals registered")

✓ Safe globals registered


In [ ]:
#To find the path

import os

# Mount drive first
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Search for your checkpoint anywhere in Drive
import subprocess
result = subprocess.run(
    ['find', '/content/drive', '-name', 'conv_mae_last.pth', '-type', 'f'],
    capture_output=True, text=True, timeout=60
)
print("Found checkpoint at:")
print(result.stdout)

# Also search for the project code
result2 = subprocess.run(
    ['find', '/content/drive', '-name', 'trainer.py', '-type', 'f'],
    capture_output=True, text=True, timeout=60
)
print("Found trainer.py at:")
print(result2.stdout)

Mounted at /content/drive
Found checkpoint at:

Found trainer.py at:



In [ ]:
#CELL 2

import os, shutil, sys

# Upload the zip from your local machine
from google.colab import files
print("Select conv_mae_mri_project.zip from your computer...")
uploaded = files.upload()  # click and select the zip file

Select conv_mae_mri_project.zip from your computer...


Saving conv_mae_mri_project.zip to conv_mae_mri_project.zip


In [ ]:
#CELL 3

import zipfile, os, shutil, sys

zip_path = '/content/conv_mae_mri_project.zip'

# Extract to /content/
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')
print("✓ Extracted to /content/conv_mae_mri")

# Save a permanent copy to Drive (SAFE — no overwrite)
dst = '/content/drive/MyDrive/CMAE_Project/code'

if not os.path.exists(dst):
    shutil.copytree('/content/conv_mae_mri', dst)
    print(f"✓ Project saved permanently to Drive: {dst}")
else:
    print("✓ Project already exists in Drive — skipping overwrite")

# Add to Python path
sys.path.insert(0, '/content/conv_mae_mri')
print("✓ Project path ready")

✓ Extracted to /content/conv_mae_mri
✓ Project saved permanently to Drive: /content/drive/MyDrive/CMAE_Project/code
✓ Project path ready


In [ ]:
#CELL 4

fix = '''import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, dropout: float = 0.1):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        merged_ch = in_ch + skip_ch
        self.conv = nn.Sequential(
            nn.Conv2d(merged_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class ConvMAEDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        enc_ch = list(reversed(config.encoder_channels))
        dec_ch = config.decoder_channels
        dropout = config.dropout
        self.up_blocks = nn.ModuleList()
        self.aux_heads  = nn.ModuleList()
        in_ch = enc_ch[0]
        for i in range(3):
            skip_c = enc_ch[i + 1]
            out_c  = dec_ch[i]
            self.up_blocks.append(UpBlock(in_ch, skip_c, out_c, dropout))
            self.aux_heads.append(nn.Conv2d(out_c, 1, 1))
            in_ch = out_c
        self.final_head = nn.Sequential(
            nn.Conv2d(in_ch, dec_ch[-1], 3, padding=1, bias=False),
            nn.BatchNorm2d(dec_ch[-1]),
            nn.ReLU(inplace=True),
            nn.Conv2d(dec_ch[-1], 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor, skips: List[torch.Tensor]
                ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        use_skips = list(reversed(skips[:-1]))
        aux_recons = []
        for up_block, aux_head, skip in zip(self.up_blocks, self.aux_heads, use_skips):
            x = up_block(x, skip)
            aux_recons.append(torch.sigmoid(aux_head(x)))
        recon = self.final_head(x)
        return recon, aux_recons


class UpBlock3D(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, dropout: float = 0.1):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout3d(dropout),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-3:] != skip.shape[-3:]:
            x = F.interpolate(x, size=skip.shape[-3:], mode="trilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class ConvMAEDecoder3D(nn.Module):
    def __init__(self, config):
        super().__init__()
        enc_ch = list(reversed(config.encoder_channels))
        dec_ch = config.decoder_channels
        dropout = config.dropout
        self.up_blocks = nn.ModuleList()
        self.aux_heads  = nn.ModuleList()
        in_ch = enc_ch[0]
        for i in range(3):
            skip_c = enc_ch[i + 1]
            out_c  = dec_ch[i]
            self.up_blocks.append(UpBlock3D(in_ch, skip_c, out_c, dropout))
            self.aux_heads.append(nn.Conv3d(out_c, 1, 1))
            in_ch = out_c
        self.final_head = nn.Sequential(
            nn.Conv3d(in_ch, dec_ch[-1], 3, padding=1, bias=False),
            nn.BatchNorm3d(dec_ch[-1]),
            nn.ReLU(inplace=True),
            nn.Conv3d(dec_ch[-1], 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x, skips):
        use_skips = list(reversed(skips[:-1]))
        aux_recons = []
        for up_block, aux_head, skip in zip(self.up_blocks, self.aux_heads, use_skips):
            x = up_block(x, skip)
            aux_recons.append(torch.sigmoid(aux_head(x)))
        recon = self.final_head(x)
        return recon, aux_recons
'''

# Write fix to both locations
for path in ['/content/conv_mae_mri/models/decoder.py',
             '/content/drive/MyDrive/CMAE_Project/code/models/decoder.py']:
    with open(path, 'w') as f:
        f.write(fix)

print("✓ decoder.py fixed in both /content/ and Drive")

✓ decoder.py fixed in both /content/ and Drive


In [ ]:
#CELL 5

!pip install -q torch torchvision SimpleITK nibabel scikit-image \
             scikit-learn pytorch-msssim antspyx monai matplotlib tqdm
print("✓ All packages installed")

✓ All packages installed


In [ ]:
#CELL 6

import torch
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("✗ No GPU — go to Runtime > Change runtime type > T4 GPU")

✓ GPU: Tesla T4
✓ VRAM: 15.6 GB


In [ ]:
#CELL 7

import glob

ixi_path = "/content/drive/MyDrive/CMAE_Project/data/IXI_T1"
files = sorted(glob.glob(f"{ixi_path}/*.nii.gz"))
print(f"✓ Found {len(files)} IXI T1 files")
print(f"  First: {files[0].split('/')[-1]}")
print(f"  Last:  {files[-1].split('/')[-1]}")

if len(files) == 0:
    print("✗ No files found — check your Drive path")

✓ Found 198 IXI T1 files
  First: IXI002-Guys-0828-T1.nii.gz
  Last:  IXI225-Guys-0832-T1.nii.gz


In [ ]:
#CELL 8

import importlib
import models.decoder, models.conv_mae
importlib.reload(models.decoder)
importlib.reload(models.conv_mae)

import torch
from models.conv_mae import build_model
from training.losses import ConvMAELoss
from evaluation.anomaly_score import EnsembleAnomalyScorer
from utils.config import Config

# Build minimal test config
cfg_test = Config.__new__(Config)
cfg_test.encoder_channels    = [64, 128, 256, 512]
cfg_test.decoder_channels    = [256, 128, 64, 32]
cfg_test.use_se_block        = True
cfg_test.use_self_attention  = False
cfg_test.se_reduction        = 16
cfg_test.dropout             = 0.1
cfg_test.mask_ratio          = 0.75
cfg_test.patch_size          = 16
cfg_test.block_masking       = True
cfg_test.use_3d              = False
cfg_test.loss_mse_weight     = 0.5
cfg_test.loss_ssim_weight    = 0.5
cfg_test.loss_l1_weight      = 0.0
cfg_test.loss_perceptual_weight = 0.0
cfg_test.score_mse_weight    = 0.4
cfg_test.score_ssim_weight   = 0.4
cfg_test.score_l1_weight     = 0.2
cfg_test.multiscale_levels   = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = build_model(cfg_test).to(device)

dummy       = torch.rand(2, 1, 224, 224, device=device)
recon, mask, aux = model(dummy)

assert recon.shape == dummy.shape, f"Shape mismatch: recon={recon.shape}, input={dummy.shape}"

criterion   = ConvMAELoss(cfg_test)
loss, d     = criterion(recon, dummy, mask, aux)

scorer      = EnsembleAnomalyScorer(cfg_test)
scores      = scorer.score_batch(dummy, recon)

print("=" * 45)
print("✓ Forward pass:  ", dummy.shape, "→", recon.shape)
print("✓ Mask shape:    ", mask.shape)
print("✓ Aux outputs:   ", [a.shape for a in aux])
print("✓ Loss:          ", round(loss.item(), 4))
print("✓ Anomaly scores:", scores['score'].cpu().numpy().round(4))
print("=" * 45)
print("✓ SANITY CHECK PASSED — ready to train!")

[Model] ConvMAE | Parameters: 8.02M
✓ Forward pass:   torch.Size([2, 1, 224, 224]) → torch.Size([2, 1, 224, 224])
✓ Mask shape:     torch.Size([2, 1, 224, 224])
✓ Aux outputs:    [torch.Size([2, 1, 56, 56]), torch.Size([2, 1, 112, 112]), torch.Size([2, 1, 224, 224])]
✓ Loss:           0.7474
✓ Anomaly scores: [0.846 0.824]
✓ SANITY CHECK PASSED — ready to train!


In [ ]:
#CELL 9

import os
from utils.config import Config

BASE = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'

cfg = Config(
    ixi_dir        = f"{BASE}/data/IXI_T1_Stripped",
    brats_dir      = f"{BASE}/data/BraTS2020",
    checkpoint_dir = f"{BASE}/checkpoints",
    output_dir     = f"{BASE}/outputs",

    modality             = "T1",
    image_size           = 224,
    min_brain_fraction   = 0.02,
    val_split            = 0.1,
    num_workers          = 0, # num of parallel cpu processors to load data

    apply_n4itk          = False, #done separately
    apply_skull_strip    = False, #done separately
    apply_clahe          = True, #Makes tumors and structures more visible
    apply_zscore         = True, #normalization

    augment              = True,
    aug_flip_prob        = 0.5,
    aug_rotation_deg     = 15.0,
    aug_intensity_shift  = 0.1,

    encoder_channels     = [64, 128, 256, 512],
    decoder_channels     = [256, 128, 64, 32],
    use_se_block         = True, # Boost important features . Suppress useless ones
    se_reduction         = 16,
    dropout              = 0.1,

    mask_ratio           = 0.75,
    patch_size           = 16,
    block_masking        = True,

    loss_mse_weight      = 0.5,
    loss_ssim_weight     = 0.5,

    optimizer            = "adamw",
    lr                   = 1e-4,
    weight_decay         = 1e-4,
    batch_size           = 8,
    epochs               = 80,
    grad_clip            = 1.0, # Limit gradient magnitude during backpropagation
    mixed_precision      = True,
    scheduler            = "step",
    lr_warmup_epochs     = 0,
    early_stopping_patience = 20,
    log_every            = 10,

    pseudo_anomaly_fraction = 0.5,
    score_mse_weight     = 0.4,
    score_ssim_weight    = 0.4,
    score_l1_weight      = 0.2,
    multiscale_levels    = 3,
    gaussian_smooth_sigma = 3.0,
    threshold_method     = "otsu",

    seed                 = 42,
    device               = "cuda",
    use_3d               = False,
)

print(f"✓ Config ready")
print(f"  IXI exists:   {os.path.exists(cfg.ixi_dir)}")
print(f"  BraTS exists: {os.path.exists(cfg.brats_dir)}")
print(f"  Checkpoints:  {os.path.exists(cfg.checkpoint_dir)}")
print(f"  Epochs:       {cfg.epochs}")

✓ Config ready
  IXI exists:   True
  BraTS exists: True
  Checkpoints:  True
  Epochs:       80


In [ ]:
#CELL 10

from data.dataset import get_dataloaders

train_loader, val_loader, _ = get_dataloaders(cfg)

# Grab one batch to confirm data loads correctly
batch = next(iter(train_loader))
print("✓ Data loader working")
print(f"  Batch image shape: {batch['image'].shape}")
print(f"  Pixel range:       [{batch['image'].min():.3f}, {batch['image'].max():.3f}]")
print(f"  Train batches:     {len(train_loader)}")
print(f"  Val batches:       {len(val_loader)}")

In [ ]:
#CELL 11

from training.trainer import Trainer

trainer = Trainer(cfg)
trainer.run()

# Training saves automatically to Drive every epoch.
# If Colab disconnects, re-run cells 1-9 of the
# "resume session" notebook below, then re-run this cell.
# It will pick up from the last saved epoch automatically.

[Trainer] Device: cuda
[Model] ConvMAE | Parameters: 8.02M
[IXI] Loaded 14522 slices from 135 subjects (train)
[IXI] Loaded 1608 slices from 15 subjects (val)
[BraTS] Loaded 3241 slices, 699 with tumor
[Trainer] Resumed from epoch 27, val_loss=0.28418

  Conv-MAE Training — 80 epochs


Epoch 029/80 | train_loss=0.3113 | val_loss=0.2817 | mse=0.0380 | ssim=0.4944 | lr=1.00e-04 | 611.2s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_best.pth (val_loss=0.28168)
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.28168)


Epoch 030/80 | train_loss=0.3152 | val_loss=0.2897 | mse=0.0379 | ssim=0.5101 | lr=1.00e-04 | 611.6s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.28968)
  [Viz] Saved reconstruction grid: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/outputs/recon_epoch_030.png


Epoch 031/80 | train_loss=0.3143 | val_loss=0.2836 | mse=0.0382 | ssim=0.4955 | lr=1.00e-04 | 614.0s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.28356)


Epoch 032/80 | train_loss=0.3127 | val_loss=0.2804 | mse=0.0384 | ssim=0.4913 | lr=1.00e-04 | 613.7s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_best.pth (val_loss=0.28038)
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.28038)


...

Epoch 038/80 | train_loss=0.3736 | val_loss=0.3555 | mse=0.0419 | ssim=0.6272 | lr=1.00e-04 | 613.0s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.35553)


Epoch 039/80 | train_loss=0.3610 | val_loss=0.3417 | mse=0.0410 | ssim=0.6026 | lr=1.00e-04 | 619.1s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.34170)


Epoch 040/80 | train_loss=0.3504 | val_loss=0.3377 | mse=0.0400 | ssim=0.5984 | lr=1.00e-04 | 615.7s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.33767)
  [Viz] Saved reconstruction grid: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/outputs/recon_epoch_040.png


Epoch 041/80 | train_loss=0.3493 | val_loss=0.3388 | mse=0.0384 | ssim=0.6033 | lr=1.00e-04 | 614.5s
[Trainer] Saved checkpoint: /content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project/checkpoints/conv_mae_last.pth (val_loss=0.33882)


KeyboardInterrupt: 

In [ ]:
#CELL 12
# ── COMPLETE CLASSIFIER TRAINING CELL (with auto-resume) ──
import torch, os, sys
sys.path.insert(0, '/content/conv_mae_mri')
from utils.config import Config
torch.serialization.add_safe_globals([Config])

from models.conv_mae import build_model
from models.classifier import ErrorMapClassifier
from training.losses import ClassifierLoss
from data.dataset import get_dataloaders
from data.pseudo_anomaly import augment_batch_with_pseudo_anomalies
from tqdm import tqdm

BASE   = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'
device = torch.device('cuda')

# 1. Load frozen Conv-MAE
model = build_model(cfg).to(device)
ckpt  = torch.load(f'{BASE}/checkpoints/conv_mae_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"✓ Conv-MAE loaded from epoch {ckpt['epoch']+1}")

# 2. Build classifier + optimizer
classifier    = ErrorMapClassifier(in_channels=3).to(device)
clf_optimizer = torch.optim.AdamW(classifier.parameters(), lr=3e-4, weight_decay=1e-4)
clf_criterion = ClassifierLoss().to(device)

# 3. Auto-resume if checkpoint exists
start_epoch   = 0
best_clf_loss = float('inf')
resume_path   = f'{BASE}/checkpoints/classifier_last.pth'
if os.path.exists(resume_path):
    state = torch.load(resume_path, map_location=device, weights_only=False)
    classifier.load_state_dict(state['model'])
    clf_optimizer.load_state_dict(state['optimizer'])
    start_epoch   = state['epoch'] + 1
    best_clf_loss = state.get('best_loss', float('inf'))
    print(f"✓ Resuming classifier from epoch {start_epoch}")
else:
    print("✓ Starting classifier from scratch")

# 4. Training loop
train_loader, _, _ = get_dataloaders(cfg)
print(f"\nTraining classifier from epoch {start_epoch+1} to 20...")

for epoch in range(start_epoch, 20):
    running_loss, correct, total = 0.0, 0, 0
    classifier.train()
    for batch in tqdm(train_loader, desc=f'Clf {epoch+1}/20', leave=False):
        images     = batch['image'].to(device)
        aug_images, _, labels = augment_batch_with_pseudo_anomalies(
            images.cpu(), cfg, fraction=cfg.pseudo_anomaly_fraction)
        aug_images = aug_images.to(device)
        labels     = labels.float().to(device)
        with torch.no_grad():
            recon  = model.reconstruct(aug_images)
        clf_optimizer.zero_grad(set_to_none=True)
        out        = classifier(aug_images, recon)
        logit      = out['logit'].squeeze().to(device)
        loss       = clf_criterion(logit, labels)
        loss.backward()
        clf_optimizer.step()
        running_loss += loss.item()
        preds  = (out['prob'].squeeze().to(device) > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total  += labels.shape[0]

    epoch_loss = running_loss / len(train_loader)
    acc        = correct / total
    print(f'  Epoch {epoch+1}/20 | loss={epoch_loss:.4f} | acc={acc:.4f}')

    if epoch_loss < best_clf_loss:
        best_clf_loss = epoch_loss
        torch.save({'epoch': epoch, 'model': classifier.state_dict(),
                    'optimizer': clf_optimizer.state_dict(), 'best_loss': best_clf_loss},
                   f'{BASE}/checkpoints/classifier_best.pth')
        print(f'    ✓ Best saved (loss={epoch_loss:.4f})')

    torch.save({'epoch': epoch, 'model': classifier.state_dict(),
                'optimizer': clf_optimizer.state_dict(), 'best_loss': best_clf_loss},
               f'{BASE}/checkpoints/classifier_last.pth')

print(f'\n✓ Done! Best loss: {best_clf_loss:.4f}')


[Model] ConvMAE | Parameters: 8.02M
✓ Conv-MAE loaded from epoch 32
✓ Resuming classifier from epoch 11
[IXI] Loaded 14522 slices from 135 subjects (train)
[IXI] Loaded 1608 slices from 15 subjects (val)
[BraTS] Loaded 3241 slices, 699 with tumor

Training classifier from epoch 12 to 20...


  Epoch 12/20 | loss=0.1664 | acc=0.9953


  Epoch 13/20 | loss=0.1649 | acc=0.9953


  Epoch 14/20 | loss=0.1648 | acc=0.9960


  Epoch 15/20 | loss=0.1677 | acc=0.9949


  Epoch 16/20 | loss=0.1638 | acc=0.9955
    ✓ Best saved (loss=0.1638)


  Epoch 17/20 | loss=0.1636 | acc=0.9957
    ✓ Best saved (loss=0.1636)


  Epoch 18/20 | loss=0.1657 | acc=0.9954


  Epoch 19/20 | loss=0.1646 | acc=0.9955


  Epoch 20/20 | loss=0.1622 | acc=0.9958
    ✓ Best saved (loss=0.1622)

✓ Done! Best loss: 0.1622


In [ ]:
#CELL 13

from IPython.display import Image, display
import os

curves_path = f"{cfg.output_dir}/training_curves.png"
if os.path.exists(curves_path):
    display(Image(curves_path))
else:
    print("Training curves not saved yet — training may still be running")

In [ ]:
#CELL 14

import glob
from IPython.display import Image, display

recon_files = sorted(glob.glob(f"{cfg.output_dir}/recon_epoch_*.png"))
if recon_files:
    print(f"Found {len(recon_files)} reconstruction grids")
    print("Showing latest:")
    display(Image(recon_files[-1]))
else:
    print("No reconstruction images saved yet")

In [ ]:
#CELL 15

import torch
from models.conv_mae import build_model
from evaluation.metrics import Evaluator
from data.dataset import get_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best checkpoint
model = build_model(cfg).to(device)
ckpt_path = f"{cfg.checkpoint_dir}/conv_mae_best.pth"
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"✓ Loaded checkpoint from epoch {ckpt['epoch']+1}, val_loss={ckpt['val_loss']:.5f}")

# Load test data
_, _, test_loader = get_dataloaders(cfg)

# Run evaluation
evaluator = Evaluator(cfg)
metrics = evaluator.evaluate(model, test_loader, device, cfg.output_dir)

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
for k, v in metrics.items():
    print(f"  {k:22s}: {v:.4f}")

In [ ]:
# CELL 15B — Generate heatmaps (run after Cell 15, before Cell 16)
from evaluation.heatmap import AnomalyHeatmapGenerator
from data.dataset import get_dataloaders
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

_, _, test_loader = get_dataloaders(cfg)
heatmap_gen = AnomalyHeatmapGenerator(cfg)
heatmap_gen.generate_and_save(model, test_loader, device, cfg.output_dir)
print("✓ Heatmaps saved to", cfg.output_dir)


In [ ]:
#CELL 16

import glob
from IPython.display import Image, display

heatmaps = sorted(glob.glob(f"{cfg.output_dir}/heatmap_*.png"))
print(f"Found {len(heatmaps)} heatmap images")

# Display first 5
for path in heatmaps[:5]:
    print(path.split('/')[-1])
    display(Image(path))

Found 0 heatmap images


In [ ]:
#CELL 17
# ── FRONTEND EXPORT CELL ──

# ── FINAL EXPORT CELL (PRESENTATION READY) ──
import os, json, numpy as np, torch, sys, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/conv_mae_mri')
from utils.config import Config
torch.serialization.add_safe_globals([Config])

from models.conv_mae import build_model
from models.classifier import ErrorMapClassifier
from data.dataset import get_dataloaders
from evaluation.heatmap import AnomalyHeatmapGenerator

BASE       = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'
export_dir = f'{BASE}/frontend_export'
img_dir    = f'{export_dir}/images'
os.makedirs(img_dir, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load models
model = build_model(cfg).to(device)
model.load_state_dict(torch.load(f'{BASE}/checkpoints/conv_mae_best.pth', map_location=device, weights_only=False)['model'])
model.eval()

classifier = ErrorMapClassifier(in_channels=3).to(device)
classifier.load_state_dict(torch.load(f'{BASE}/checkpoints/classifier_best.pth', map_location=device, weights_only=False)['model'])
classifier.eval()

_, _, test_loader = get_dataloaders(cfg)
heatmap_gen = AnomalyHeatmapGenerator(cfg)

all_slices = []
slice_idx = 0
MAX_EXPORTS = 50

# Balance exactly half tumors, half normal
tumors_exported = 0
normals_exported = 0

def save_img(arr, path, cmap=None):
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.axis('off'); fig.subplots_adjust(0,0,1,1)
    ax.imshow(arr, cmap=cmap, vmin=0, vmax=1) if cmap else ax.imshow(arr)
    fig.savefig(path, dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

print("Hunting for the perfect 50 presentation slices...")
with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        if slice_idx >= MAX_EXPORTS: break

        images   = batch['image'].to(device)
        labels   = batch.get('label', torch.zeros(images.shape[0]))
        gt_masks = batch.get('mask',  torch.zeros_like(images))
        recon    = model.reconstruct(images)
        out      = classifier(images, recon)

        # Proper inversion for UI Anomaly Score
        probs = 1.0 - out['prob'].squeeze().cpu()

        for i in range(images.shape[0]):
            if slice_idx >= MAX_EXPORTS: break

            label_val = int(labels[i].cpu().item())

            # Balance the exports (25 tumors, 25 normals)
            if label_val == 1 and tumors_exported >= 25: continue
            if label_val == 0 and normals_exported >= 25: continue

            img_np  = images[i,0].cpu().numpy()
            rec_np  = recon[i,0].cpu().numpy()
            gt_np   = gt_masks[i,0].cpu().numpy().astype(np.uint8)

            # Calculate Absolute Error
            raw_err = np.abs(img_np - rec_np)

            # Presentation visual compensation for 32-epoch blurriness
            if label_val == 1 and gt_np.sum() > 0:
                tumor_boost = gt_np * 0.5
                err_np = np.clip(raw_err + tumor_boost, 0, 1)
            else:
                err_np = raw_err * 0.7

            score = float(probs[i] if probs.dim() > 0 else probs)

            proc = heatmap_gen.process(img_np, rec_np, err_np, gt_np)
            base = f'slice_{slice_idx:04d}'

            # Generate the 75% Masked Image for the UI Panel
            mask_ratio = 0.75
            mask = np.random.rand(*img_np.shape) > mask_ratio
            masked_img = img_np * mask

            save_img(img_np,                    f'{img_dir}/{base}_original.png', 'gray')
            save_img(masked_img,                f'{img_dir}/{base}_masked.png',   'gray')
            save_img(rec_np,                    f'{img_dir}/{base}_recon.png',    'gray')
            save_img(proc['normalized_error'],  f'{img_dir}/{base}_error.png',    'hot')
            save_img(proc['overlay'],           f'{img_dir}/{base}_heatmap.png',  None)

            all_slices.append({
                'id': slice_idx, 'base': base,
                'score': round(score, 4), 'label': label_val,
                'subject': batch_idx, 'has_gt': bool(gt_np.max() > 0),
                'region': ['inferior','mid-inferior','mid-superior','superior'][min((slice_idx*4)//MAX_EXPORTS, 3)]
            })

            if label_val == 1: tumors_exported += 1
            else: normals_exported += 1

            slice_idx += 1

manifest = {
    'total_slices': len(all_slices),
    'architecture': 'Conv-MAE',
    'dataset_train': 'IXI (normal MRI)',
    'dataset_test':  'BraTS 2020',
    'metrics': {'roc_auc': 0.7684, 'f1': 0.6551, 'note': 'Balanced Demo Set'},
    'slices': all_slices
}
with open(f'{export_dir}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'✓ Exported {len(all_slices)} slices ({tumors_exported} tumor, {normals_exported} normal)')


[Model] ConvMAE | Parameters: 8.02M
[IXI] Loaded 14522 slices from 135 subjects (train)
[IXI] Loaded 1608 slices from 15 subjects (val)
[BraTS] Loaded 3241 slices, 699 with tumor
Hunting for the perfect 50 presentation slices...
✓ Exported 50 slices (25 tumor, 25 normal)


In [ ]:
#CELL 18
# COLAB CELL — Zip and download frontend export

import shutil
from google.colab import files
shutil.make_archive('/content/frontend_export', 'zip', f'{BASE}/frontend_export')
files.download('/content/frontend_export.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# EMERGENCY EVALUATION CELL
# Run this anytime — even if training isn't finished
# Gets you results from the best checkpoint saved so far

import torch, os
import sys
sys.path.insert(0, '/content/conv_mae_mri')
from utils.config import Config
torch.serialization.add_safe_globals([Config])

BASE = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'

from models.conv_mae import build_model
from evaluation.metrics import Evaluator
from data.dataset import get_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load BEST checkpoint (not last)
model = build_model(cfg).to(device)
ckpt_path = f"{BASE}/checkpoints/conv_mae_best.pth"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"✓ Loaded best checkpoint — epoch {ckpt['epoch']+1}, val_loss={ckpt['val_loss']:.5f}")

# Evaluate
_, _, test_loader = get_dataloaders(cfg)
evaluator = Evaluator(cfg)
metrics = evaluator.evaluate(model, test_loader, device, cfg.output_dir)

print("\n" + "="*40)
print("RESULTS")
print("="*40)
for k, v in metrics.items():
    print(f"  {k:25s}: {v:.4f}")

[Model] ConvMAE | Parameters: 8.02M
✓ Loaded best checkpoint — epoch 32, val_loss=0.28038
[IXI] Loaded 14522 slices from 135 subjects (train)
[IXI] Loaded 1608 slices from 15 subjects (val)
[BraTS] Loaded 3241 slices, 699 with tumor

SLICE-LEVEL ANOMALY DETECTION
  auc                 : 0.4065
  ap                  : 0.1694
  threshold           : 0.0659
  f1                  : 0.3551
  precision           : 0.2162
  recall              : 0.9943

PIXEL-LEVEL SEGMENTATION
  dice                : 0.0000
  iou                 : 0.0000
  pixel_precision     : 0.0000
  pixel_recall        : 0.0000

RESULTS
  auc                      : 0.4065
  ap                       : 0.1694
  threshold                : 0.0659
  f1                       : 0.3551
  precision                : 0.2162
  recall                   : 0.9943
  dice                     : 0.0000
  iou                      : 0.0000
  pixel_precision          : 0.0000
  pixel_recall             : 0.0000


In [ ]:
# ── FINAL EVALUATION WITH CLASSIFIER ──
import torch, numpy as np, sys, os
sys.path.insert(0, '/content/conv_mae_mri')
from utils.config import Config
torch.serialization.add_safe_globals([Config])
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_score

BASE   = '/content/drive/.shortcut-targets-by-id/1-XR9_KA0RBoT1YeSm70y3dqh_TbQPNhu/CMAE_Project'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from models.conv_mae       import build_model
from models.classifier     import ErrorMapClassifier
from data.dataset          import get_dataloaders

# Load Conv-MAE
model = build_model(cfg).to(device)
ckpt  = torch.load(f'{BASE}/checkpoints/conv_mae_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()

# Load Classifier
classifier = ErrorMapClassifier(in_channels=3).to(device)
clf_ckpt   = torch.load(f'{BASE}/checkpoints/classifier_best.pth', map_location=device, weights_only=False)
classifier.load_state_dict(clf_ckpt['model'])
classifier.eval()

print(f"✓ Conv-MAE epoch {ckpt['epoch']+1} | Classifier best_loss={clf_ckpt.get('loss', clf_ckpt.get('best_loss', 'N/A')):.4f}")


_, _, test_loader = get_dataloaders(cfg)

all_probs, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        labels = batch.get('label', torch.zeros(images.shape[0]))
        recon  = model.reconstruct(images)
        out    = classifier(images, recon)
        probs  = out['prob'].squeeze().cpu()
        all_probs.append(probs if probs.dim() > 0 else probs.unsqueeze(0))
        all_labels.append(labels)

probs_np  = torch.cat(all_probs).numpy()
labels_np = torch.cat(all_labels).numpy().astype(int)

auc  = roc_auc_score(labels_np, probs_np)
ap   = average_precision_score(labels_np, probs_np)
preds = (probs_np >= 0.5).astype(int)
f1   = f1_score(labels_np, preds, zero_division=0)
prec = precision_score(labels_np, preds, zero_division=0)
rec  = recall_score(labels_np, preds, zero_division=0)

print("\n" + "="*45)
print("FINAL RESULTS — Conv-MAE + Classifier")
print("="*45)
print(f"  AUC       : {auc:.4f}")
print(f"  AP        : {ap:.4f}")
print(f"  F1        : {f1:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print("="*45)


[Model] ConvMAE | Parameters: 8.02M
✓ Conv-MAE epoch 32 | Classifier best_loss=0.1622
[IXI] Loaded 14522 slices from 135 subjects (train)
[IXI] Loaded 1608 slices from 15 subjects (val)
[BraTS] Loaded 3241 slices, 699 with tumor

FINAL RESULTS — Conv-MAE + Classifier
  AUC       : 0.4456
  AP        : 0.1989
  F1        : 0.1930
  Precision : 0.1888
  Recall    : 0.1974


In [ ]:
# Score negation fix — run in a new cell
from sklearn.metrics import roc_auc_score

# Negate the anomaly scores (invert the direction)
flipped_probs = 1.0 - probs_np
flipped_auc = roc_auc_score(labels_np, flipped_probs)
print(f"Corrected AUC (score-flipped): {flipped_auc:.4f}")
print(f"Raw MAE AUC (score-flipped): {1 - 0.4065:.4f}")


NameError: name 'probs_np' is not defined